# 🧱 Data Forge — quick tour

Shows how to ingest mixed documents (PDF, DOCX, XLSX, images + OCR...) and build a training dataset with the correct chat template for **any** domain you care about.

The notebook is **not tied to `asset_integrity`** — set `DOMAIN_NAME` to whatever engagement you're working on.

Run from the repo root.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('.'))

DOMAIN_NAME   = 'ai_llm'                     # ← change per engagement
BASE_MODEL    = 'Qwen/Qwen2.5-7B-Instruct'
SYSTEM_PROMPT = 'You are an expert in the chosen domain. Be precise and cite sources.'

from app.config_loader import create_domain_config, domain_config_exists, load_domain_config

if not domain_config_exists(DOMAIN_NAME):
    create_domain_config(name=DOMAIN_NAME, system_prompt=SYSTEM_PROMPT)
    print(f'✅ Created configs/domains/{DOMAIN_NAME}.yaml')

config = load_domain_config(DOMAIN_NAME)
SYSTEM_PROMPT = config['system_prompt']

In [ ]:
from app.data_forge import DataForge

forge = DataForge()
records = forge.ingest('./data/uploads/')
print(f'Ingested {len(records)} records')
for r in records[:3]:
    print(r.source_type, '-', r.source[:80], '-', len(r.text), 'chars')

In [ ]:
ds = forge.build_dataset(
    records,
    task='sft',
    base_model=BASE_MODEL,
    system_prompt=SYSTEM_PROMPT,
    synth_qa=True,
    target_size=200,
)
print(f'Rows: {len(ds)}')
print(ds[0]['text'][:800])
ds.save_to_disk(f'./data/processed/{DOMAIN_NAME}_sft')